# pyspark_intro

Pyspark is the official API for apache spark 

 short note on notebooks in databricks
- SQL notebook 
- Python notebook

mkes all cell default to that choice e.g SQL

# Infer schema

-empty strings bcos of null values

In [0]:
DATA_PATH= "/Volumes/data/olympic_games/raw_data"

df_athletes=spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)
df_athletes

In [0]:
df_athletes.limit(2).display()


In [0]:
display(df_athletes)

In [0]:
df_athletes.printSchema()

In [0]:
(df_athletes.count(),len(df_athletes.columns))

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, IntegerType, StructType, FloatType, ByteType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

# EDA

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter("NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'Isl')AND Medal != 'NA'").sort("NOC", "Medal").display()

# Spark SQL

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

spark.sql("FROM df_athletes_schema").display()

In [0]:
spark.sql("""
SELECT Sport, NOC, Medal, count(*) AS Total
FROM df_athletes_schema
WHERE Sport = 'Table Tennis' AND NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'
GROUP BY Sport, NOC, Medal
""").display()

# Ingesting data to unity catalog

In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals AS
(
    SELECT
        name,
        age,
        height,
        weight,
        year,
        sport,
        medal
    FROM
       df_athletes_schema
    WHERE
        noc = 'SWE' and medal IN ('Bronze', 'Silver', 'Gold')
)

In [0]:
%sql
SELECT name, sport, medal
FROM data.olympic_games.sweden_medals
WHERE sport = 'Swimming'
ORDER BY name